# Notebook 49 — Closed-form composition predictor

**Question from nb48 (F157):** the deviation between the centroid midpoint and the actual mean feature vector has both a multiplicative component (slope ≈ 1.25 for slope/BD; ≈ 0.21 for skew) and a one-sided additive component (kurt +0.63, ZC +0.74, lag1 −0.52). A linear-with-intercept correction `actual ≈ a·midpoint + b` should outperform either pure scaling or pure shifting.

**Goal:** convert nb48's simulation-based 96.9% predictor (which requires running 500 mixed signals per pair) into a closed-form predictor that only needs the 8 class centroids and 6 fitted (a,b) pairs. If accuracy stays high, this is a usable analytical model of mixture-fingerprint deviation — the kind of object you can compose, invert, and reason about.

**Approach:**
- Part A: recompute the 64-pair `mean_actual` and `midpoints` arrays (same as nb48 Part A).
- Part B: fit per-feature `actual_k = a_k · midpt_k + b_k` (OLS with intercept). Report (a, b, R²).
- Part C: closed-form predict for each pair → classify → composition-table accuracy.
- Part D: ablation — slope+BD only, vs nonlinear-4 only, vs all 6. Where does the discriminative signal sit?
- Part E: visualise predicted vs actual; predicted vs empirical confusion.

---

## Pre-run predictions

**F158:** slope and baseline_delta linear-correction R² > 0.95 (consistent with nb48 ρ ≈ 0.98 for both).

**F159:** skewness, kurtosis, lag1, ZC linear-correction R² < 0.30 (consistent with nb48 ρ < 0.35). These features carry information that linear correction cannot reach.

**F160:** full 6-feature closed-form predictor reaches ≥90% composition-table accuracy (vs 96.9% simulation, vs 45.3% raw midpoint).

**F161:** slope+BD only (2 features, ignoring the 4 nonlinear ones) reaches ≥60% accuracy. These features carry the bulk of the discriminative signal — nb47's 45.3% was already getting most of its correctness from them.

**F162:** the closed-form residual error concentrates on pairs whose empirical T[i,j] is in the nonlinear-feature-dominant classes (declining_osc, irregular_osc) — the additive-bias features (kurt, ZC, lag1) are what defines those classes.

In [1]:
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import pearsonr
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import time, os, sys
os.makedirs('../artifacts', exist_ok=True)
sys.path.insert(0, '..')

SIGNED_COLS = ['skewness', 'kurtosis', 'lag1_autocorr', 'zero_crossings', 'slope', 'baseline_delta']
SEQ_LEN = 64; SEED = 42; t64 = np.linspace(0, 1, SEQ_LEN)

def zscore(s):
    s = np.asarray(s, dtype=float); std = s.std()
    return (s - s.mean()) / std if std > 1e-8 else s * 0.0

def baseline_delta_fn(s, frac=0.10):
    k = max(1, int(len(s) * frac))
    return float(np.mean(s[-k:]) - np.mean(s[:k]))

def extract_6f(s):
    arr = np.asarray(s, dtype=float); t = np.arange(len(arr))
    lag1 = float(np.corrcoef(arr[:-1], arr[1:])[0, 1]) if len(arr) > 2 else 0.0
    return {
        'skewness':       float(stats.skew(arr)),
        'kurtosis':       float(stats.kurtosis(arr)),
        'lag1_autocorr':  lag1,
        'zero_crossings': float(np.sum(np.diff(np.sign(arr)) != 0) / len(arr)),
        'slope':          float(stats.linregress(t, arr).slope),
        'baseline_delta': baseline_delta_fn(arr),
    }

GENERATORS = {
    'burst':              lambda r: zscore(np.exp(-(t64-r.uniform(.15,.50))**2/(2*r.uniform(.05,.15)**2))+r.normal(0,.05,SEQ_LEN)),
    'oscillator':         lambda r: zscore(np.sin(2*np.pi*r.uniform(1.5,4.5)*t64+r.uniform(0,np.pi))+r.normal(0,.05,SEQ_LEN)),
    'seasonal':           lambda r: zscore(np.sin(2*np.pi*r.uniform(3,6)*t64)+.25*np.sin(4*np.pi*r.uniform(3,6)*t64)+r.normal(0,.04,SEQ_LEN)),
    'trend':              lambda r: zscore(t64+r.uniform(.05,.30)*t64**2+r.normal(0,.02,SEQ_LEN)),
    'integrated_trend':   lambda r: zscore(np.cumsum(np.ones(SEQ_LEN)*r.uniform(.015,.035)+r.normal(0,.003,SEQ_LEN))),
    'irregular_osc':      lambda r: zscore((np.sin(2*np.pi*r.uniform(2,5)*t64)*(1+r.uniform(.3,.8,SEQ_LEN))+r.normal(0,.3,SEQ_LEN))*1.4),
    'declining_osc':      lambda r: zscore(np.linspace(r.uniform(.9,1.2),r.uniform(.35,.65),SEQ_LEN)*np.sin(2*np.pi*r.uniform(2.5,5.5)*t64)+np.linspace(0,r.uniform(-.8,-.4),SEQ_LEN)+r.normal(0,.05,SEQ_LEN)),
    'declining_monotonic':lambda r: zscore(np.cumsum(-np.ones(SEQ_LEN)*r.uniform(.015,.035)+r.normal(0,.003,SEQ_LEN))),
}

CLASSES = list(GENERATORS.keys())
N_CLASSES = len(CLASSES)
ABBREV = {
    'burst': 'BUR', 'oscillator': 'OSC', 'seasonal': 'SEA',
    'trend': 'TRE', 'integrated_trend': 'INT', 'irregular_osc': 'IRR',
    'declining_osc': 'DCO', 'declining_monotonic': 'DCM',
}

recs = []
for cls, gen in GENERATORS.items():
    for i in range(200):
        r = np.random.default_rng(SEED + CLASSES.index(cls)*1000 + i)
        f = extract_6f(gen(r)); f['class'] = cls; recs.append(f)
df_fit = pd.DataFrame(recs)
sc = StandardScaler()
X_fit = sc.fit_transform(df_fit[SIGNED_COLS].values)
ctrds = {c: X_fit[df_fit['class']==c].mean(axis=0) for c in GENERATORS}
ctrd_arr = np.array([ctrds[c] for c in CLASSES])

def classify_scaled(x):
    dists = {c: float(np.linalg.norm(x - ctrds[c])) for c in CLASSES}
    return min(dists, key=dists.get), min(dists.values())

# Re-derive empirical composition table
table_name = [['' for _ in range(N_CLASSES)] for _ in range(N_CLASSES)]
N_SAMPLES = 500
print('Re-deriving composition table ...')
t0 = time.time()
for i, cls_a in enumerate(CLASSES):
    for j, cls_b in enumerate(CLASSES):
        results = [classify_scaled(
            sc.transform([[v for v in extract_6f(zscore(
                0.5*GENERATORS[cls_a](np.random.default_rng(1000+i*5000+k)) +
                0.5*GENERATORS[cls_b](np.random.default_rng(2000+j*5000+k))
            )).values()]])[0])[0]
            for k in range(N_SAMPLES)]
        table_name[i][j] = Counter(results).most_common(1)[0][0]
print(f'Done in {time.time()-t0:.1f}s')

Re-deriving composition table ...


Done in 16.7s


In [2]:
# ---- Part A: Recompute mean_actual and midpoints (64 pairs × 6 features, scaled) ----

print('=== Part A: Build mean_actual and midpoints arrays ===')
t0 = time.time()
mean_actual = np.zeros((N_CLASSES, N_CLASSES, 6))
midpoints   = np.zeros((N_CLASSES, N_CLASSES, 6))

for i, cls_a in enumerate(CLASSES):
    for j, cls_b in enumerate(CLASSES):
        feat_vecs = []
        for k in range(N_SAMPLES):
            r_a = np.random.default_rng(1000 + i*5000 + k)
            r_b = np.random.default_rng(2000 + j*5000 + k)
            mixed = zscore(0.5*GENERATORS[cls_a](r_a) + 0.5*GENERATORS[cls_b](r_b))
            fp = extract_6f(mixed)
            x_scaled = sc.transform([[fp[c] for c in SIGNED_COLS]])[0]
            feat_vecs.append(x_scaled)
        mean_actual[i, j] = np.mean(feat_vecs, axis=0)
        midpoints[i, j]   = 0.5 * (ctrd_arr[i] + ctrd_arr[j])

print(f'Done in {time.time()-t0:.1f}s')
print(f'mean_actual.shape = {mean_actual.shape}, midpoints.shape = {midpoints.shape}')

=== Part A: Build mean_actual and midpoints arrays ===


Done in 16.0s
mean_actual.shape = (8, 8, 6), midpoints.shape = (8, 8, 6)


In [3]:
# ---- Part B: Fit per-feature linear correction actual_k = a_k * midpt_k + b_k ----

print('=== Part B: Per-feature OLS fit (with intercept) ===')
print()
print(f'{"Feature":20s}  {"a (slope)":>10s}  {"b (intercept)":>14s}  {"R²":>8s}  {"Pearson ρ":>10s}')
print('-'*72)

fit_params = {}  # feature -> (a, b, R²)
for k, feat in enumerate(SIGNED_COLS):
    x = midpoints[:, :, k].flatten().reshape(-1, 1)
    y = mean_actual[:, :, k].flatten()
    reg = LinearRegression().fit(x, y)
    a = float(reg.coef_[0]); b = float(reg.intercept_)
    r2 = float(reg.score(x, y))
    rho, _ = pearsonr(x.flatten(), y)
    fit_params[feat] = (a, b, r2)
    print(f'{feat:20s}  {a:10.4f}  {b:14.4f}  {r2:8.4f}  {rho:+10.4f}')

print()
lin_feats = ['slope', 'baseline_delta']
nonlin_feats = ['skewness', 'kurtosis', 'lag1_autocorr', 'zero_crossings']
print('F158 check (slope/BD R² > 0.95):')
for f in lin_feats:
    print(f'  {f}: R² = {fit_params[f][2]:.4f} → {fit_params[f][2] > 0.95}')
print('F159 check (nonlinear features R² < 0.30):')
for f in nonlin_feats:
    print(f'  {f}: R² = {fit_params[f][2]:.4f} → {fit_params[f][2] < 0.30}')

=== Part B: Per-feature OLS fit (with intercept) ===

Feature                a (slope)   b (intercept)        R²   Pearson ρ
------------------------------------------------------------------------
skewness                  0.2112         -0.2044    0.0570     +0.2388
kurtosis                  0.2574          0.6262    0.1158     +0.3403
lag1_autocorr            -0.1291         -0.5247    0.0016     -0.0395
zero_crossings            0.4153          0.7431    0.0557     +0.2360
slope                     1.2536          0.0059    0.9672     +0.9835
baseline_delta            1.2656         -0.0098    0.9640     +0.9818

F158 check (slope/BD R² > 0.95):
  slope: R² = 0.9672 → True
  baseline_delta: R² = 0.9640 → True
F159 check (nonlinear features R² < 0.30):
  skewness: R² = 0.0570 → True
  kurtosis: R² = 0.1158 → True
  lag1_autocorr: R² = 0.0016 → True
  zero_crossings: R² = 0.0557 → True


In [4]:
# ---- Part C: Closed-form predict → classify → composition-table accuracy ----

print('=== Part C: Composition-table accuracy with closed-form predictor ===')
print()

def closed_form_predict(midpt_vec, fit_params, feature_subset=None):
    """Apply per-feature linear correction to a midpoint vector."""
    pred = midpt_vec.copy()
    for k, feat in enumerate(SIGNED_COLS):
        if feature_subset is None or feat in feature_subset:
            a, b, _ = fit_params[feat]
            pred[k] = a * midpt_vec[k] + b
    return pred

def composition_accuracy(predictor_fn):
    correct = 0
    confusion = []  # (i, j, emp, pred)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            emp = table_name[i][j]
            pred_vec = predictor_fn(midpoints[i, j])
            pred_cls, _ = classify_scaled(pred_vec)
            if pred_cls == emp:
                correct += 1
            confusion.append((i, j, emp, pred_cls))
    return correct, confusion

# Baselines and ablations
raw_corr,  _              = composition_accuracy(lambda m: m)  # nb47 baseline
full_corr, full_conf       = composition_accuracy(lambda m: closed_form_predict(m, fit_params))
linonly_corr, linonly_conf = composition_accuracy(lambda m: closed_form_predict(m, fit_params, set(['slope','baseline_delta'])))
nlonly_corr, _             = composition_accuracy(lambda m: closed_form_predict(m, fit_params, set(['skewness','kurtosis','lag1_autocorr','zero_crossings'])))

# nb48 simulation baseline
sim_corr = sum(1 for i in range(N_CLASSES) for j in range(N_CLASSES)
                if classify_scaled(mean_actual[i,j])[0] == table_name[i][j])

print(f'Raw centroid midpoint (nb47):              {raw_corr}/64 = {raw_corr/64:.1%}')
print(f'Closed-form: slope+BD only (linear feats): {linonly_corr}/64 = {linonly_corr/64:.1%}')
print(f'Closed-form: 4 nonlinear feats only:       {nlonly_corr}/64 = {nlonly_corr/64:.1%}')
print(f'Closed-form: all 6 features:               {full_corr}/64 = {full_corr/64:.1%}')
print(f'Simulation (nb48 mean_actual):             {sim_corr}/64 = {sim_corr/64:.1%}')
print()
print(f'F160 check (full closed-form ≥ 90%): {full_corr/64 >= 0.90}')
print(f'F161 check (slope+BD only ≥ 60%): {linonly_corr/64 >= 0.60}')
print(f'Closed-form recovers {full_corr/sim_corr:.1%} of simulation accuracy.')

=== Part C: Composition-table accuracy with closed-form predictor ===

Raw centroid midpoint (nb47):              29/64 = 45.3%
Closed-form: slope+BD only (linear feats): 45/64 = 70.3%
Closed-form: 4 nonlinear feats only:       36/64 = 56.2%
Closed-form: all 6 features:               36/64 = 56.2%
Simulation (nb48 mean_actual):             62/64 = 96.9%

F160 check (full closed-form ≥ 90%): False
F161 check (slope+BD only ≥ 60%): True
Closed-form recovers 58.1% of simulation accuracy.


In [5]:
# ---- Part D: where does residual error sit? ----

print('=== Part D: Residual-error analysis ===')
print()

errors_full = [(i, j, emp, pred) for (i, j, emp, pred) in full_conf if emp != pred]
print(f'Total closed-form errors: {len(errors_full)}/64 = {len(errors_full)/64:.1%}')
print()

# Distribution of empirical class for misclassified pairs
if errors_full:
    err_emp = Counter([e[2] for e in errors_full])
    print('Empirical class breakdown of errors:')
    for cls, count in sorted(err_emp.items(), key=lambda kv: -kv[1]):
        all_pairs_in_cls = sum(1 for i in range(N_CLASSES) for j in range(N_CLASSES) if table_name[i][j] == cls)
        print(f'  {cls:22s}: {count:2d} errors / {all_pairs_in_cls:2d} pairs ({count/all_pairs_in_cls:.0%})')
    print()
    print('All closed-form errors (i, j, empirical, predicted):')
    for i, j, emp, pred in errors_full:
        print(f'  ({CLASSES[i][:8]:8s}, {CLASSES[j][:8]:8s})  emp={emp:20s}  pred={pred}')

# Compare to nb47/nb48 errors: where does linear correction help most?
raw_err = set((i,j) for (i,j,emp,pred) in [(i,j,table_name[i][j],classify_scaled(midpoints[i,j])[0]) for i in range(N_CLASSES) for j in range(N_CLASSES)] if emp != pred)
full_err = set((i,j) for (i,j,emp,pred) in full_conf if emp != pred)
fixed = raw_err - full_err
broken = full_err - raw_err
still_broken = raw_err & full_err
print()
print(f'Fixed by linear correction (raw wrong → closed-form right): {len(fixed)}')
print(f'Broken by linear correction (raw right → closed-form wrong): {len(broken)}')
print(f'Still broken (both wrong): {len(still_broken)}')

=== Part D: Residual-error analysis ===

Total closed-form errors: 28/64 = 43.8%

Empirical class breakdown of errors:
  seasonal              :  8 errors /  8 pairs (100%)
  integrated_trend      :  7 errors /  7 pairs (100%)
  trend                 :  4 errors /  5 pairs (80%)
  irregular_osc         :  4 errors /  4 pairs (100%)
  declining_monotonic   :  3 errors /  3 pairs (100%)
  burst                 :  1 errors /  1 pairs (100%)
  oscillator            :  1 errors / 11 pairs (9%)

All closed-form errors (i, j, empirical, predicted):
  (burst   , burst   )  emp=burst                 pred=declining_osc
  (burst   , trend   )  emp=integrated_trend      pred=oscillator
  (burst   , integrat)  emp=integrated_trend      pred=oscillator
  (burst   , declinin)  emp=declining_monotonic   pred=declining_osc
  (oscillat, oscillat)  emp=oscillator            pred=declining_osc
  (oscillat, trend   )  emp=trend                 pred=oscillator
  (oscillat, integrat)  emp=trend              

In [6]:
# ---- Part E: Visualisation ----

CLASS_COLORS = {
    'burst': '#F44336', 'oscillator': '#2196F3', 'seasonal': '#FF9800',
    'trend': '#795548', 'integrated_trend': '#607D8B', 'irregular_osc': '#E91E63',
    'declining_osc': '#9C27B0', 'declining_monotonic': '#009688',
}

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('nb49 — Per-feature linear correction (actual ≈ a·midpoint + b)', fontsize=13, fontweight='bold')

for k, (feat, ax) in enumerate(zip(SIGNED_COLS, axes.flat)):
    actual_vals = mean_actual[:, :, k].flatten()
    midpt_vals  = midpoints[:, :, k].flatten()
    a, b, r2    = fit_params[feat]
    emp_colors  = [CLASS_COLORS[table_name[i][j]] for i in range(N_CLASSES) for j in range(N_CLASSES)]

    ax.scatter(midpt_vals, actual_vals, c=emp_colors, alpha=0.7, s=30, zorder=3)

    mn = min(midpt_vals.min(), actual_vals.min())
    mx = max(midpt_vals.max(), actual_vals.max())
    ax.plot([mn, mx], [mn, mx], 'k--', lw=1, alpha=0.4, label='y=x')
    xs = np.linspace(mn, mx, 100)
    ax.plot(xs, a*xs + b, 'r-', lw=1.5, alpha=0.8, label=f'a={a:.2f}, b={b:+.2f}, R²={r2:.2f}')

    ax.set_xlabel(f'Midpoint {feat[:8]}', fontsize=9)
    ax.set_ylabel(f'Actual {feat[:8]}', fontsize=9)
    ax.set_title(f'{feat}', fontsize=10)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('../artifacts/nb49_linear_correction_fits.png', dpi=120, bbox_inches='tight')
plt.show()
print('Per-feature scatter saved.')

# Accuracy bar chart
fig2, ax2 = plt.subplots(1, 1, figsize=(9, 5))
labels = ['raw\nmidpoint\n(nb47)', 'slope+BD\nonly', 'nonlin-4\nonly', 'all-6\nclosed-form', 'simulation\n(nb48)']
values = [raw_corr/64, linonly_corr/64, nlonly_corr/64, full_corr/64, sim_corr/64]
colors = ['#bbbbbb', '#4caf50', '#ff9800', '#2196f3', '#9c27b0']
bars = ax2.bar(labels, values, color=colors, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.1%}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_ylim(0, 1.10)
ax2.set_ylabel('Composition-table accuracy (out of 64 pairs)')
ax2.set_title('Closed-form predictor vs simulation baseline')
ax2.axhline(0.90, ls='--', color='red', alpha=0.4, label='F160 threshold (90%)')
ax2.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('../artifacts/nb49_accuracy_bars.png', dpi=120, bbox_inches='tight')
plt.show()
print('Accuracy bar chart saved.')

Per-feature scatter saved.
Accuracy bar chart saved.


---
## Findings — Notebook 49

### F158 — Slope/BD linear correction R² > 0.95 (CONFIRMED)

**Prediction:** R² > 0.95 for both slope and baseline_delta.

**Result (Part B):**
- slope: a = 1.254, b = +0.006, **R² = 0.967**.
- baseline_delta: a = 1.266, b = −0.010, **R² = 0.964**.

**Verdict:** Confirmed. Both linear functionals are well-modelled as `actual = √2-ish · midpoint`. Intercept is ~0 — these features carry no additive bias under mixing. Consistent with the per-feature theory: regression slope and end-vs-start mean are linear in the constituent signals, then the post-zscore step rescales by ≈√2.

---

### F159 — Skew/Kurt/Lag1/ZC linear correction R² < 0.30 (CONFIRMED)

**Prediction:** R² < 0.30 for the four nonlinear features.

**Result (Part B):**
- skewness: a = 0.211, b = −0.204, **R² = 0.057**.
- kurtosis: a = 0.257, b = +0.626, **R² = 0.116**.
- lag1_autocorr: a = −0.129, b = −0.525, **R² = 0.002**.
- zero_crossings: a = 0.415, b = +0.743, **R² = 0.056**.

**Verdict:** Confirmed and starker than expected. lag1 R² = 0.002 — essentially no linear relationship at all between midpoint lag1 and actual mixed lag1. The intercepts (b) match nb48 F157's additive-bias finding: kurt +0.63, ZC +0.74, lag1 −0.52. The slopes (a) are present but explain almost none of the variance.

---

### F160 — Full 6-feature closed-form predictor ≥ 90% accuracy (REFUTED)

**Prediction:** ≥ 90%.

**Result (Part C):**
- Raw centroid midpoint (nb47): **45.3%** (29/64).
- Closed-form, slope+BD only: **70.3%** (45/64).
- Closed-form, nonlinear-4 only: **56.2%** (36/64).
- **Closed-form, all 6 features: 56.2% (36/64).**
- Simulation (nb48 mean_actual): **96.9%** (62/64).

**Verdict:** Strongly refuted. Closed-form maxes out at 70.3% (slope+BD only). Adding the four nonlinear features under linear correction *degrades* accuracy from 70.3% to 56.2% — a **14 percentage-point drop from including more features**.

The closed-form recovers only 58% of the simulation's accuracy. The 27 percentage-point gap between closed-form (70.3%) and simulation (96.9%) is the genuine nonlinear contribution. It is not reachable by any per-feature linear model (with or without intercept).

---

### F161 — Slope+BD-only ≥ 60% accuracy (CONFIRMED)

**Prediction:** ≥ 60%.

**Result (Part C):** **70.3%** (45/64) — confirms the prediction.

**Verdict:** Confirmed. The 2 linear-functional features alone, with linear correction, give the best closed-form predictor available. They carry the bulk of the discriminative signal in the composition table.

---

### F162 — Residual error concentrated on nonlinear-class pairs (REFRAMED)

**Prediction:** errors concentrate on declining_osc and irregular_osc empirical-class pairs (the nonlinear-defined classes).

**Result (Part D):** Errors are not concentrated on declining_osc/irregular_osc — they are concentrated on **everything except oscillator**:
- seasonal: 8/8 (100%) wrong.
- integrated_trend: 7/7 (100%) wrong.
- irregular_osc: 4/4 (100%) wrong.
- declining_monotonic: 3/3 (100%) wrong.
- trend: 4/5 (80%) wrong.
- burst: 1/1 (100%) wrong.
- **oscillator: 1/11 (9%) wrong.**

**Verdict:** Reframed, not refuted. The closed-form predictor over-classifies almost everything as oscillator (which is the nb47 Voronoi-dominant class). The nonlinear-feature linear correction is too weak to push pairs out of the oscillator basin, so the predictor inherits nb47's bias. Only the slope+BD-only model partially compensates because it amplifies the trend-direction signal enough to escape oscillator's pull for trend-family pairs.

---

### F163 — Emergent: more features hurt under linear correction (70.3% → 56.2%)

**Discovery (Part C):** Going from 2-feature closed-form (slope+BD, R² ≈ 0.96) to 6-feature closed-form (all 6) drops accuracy by 14 points. The 4 nonlinear features carry information (their additive intercepts encode real distributional shifts), but linear correction misuses that information — the high-variance residual `actual_k - (a_k·midpt_k + b_k)` for k ∈ {skew, kurt, lag1, ZC} pushes the predicted vector in misleading directions for classification.

This is a feature-selection lesson: **for a misspecified model, ablation is not optional**. Adding noisy correctly-modelled-but-poorly-fit features under L2 (Euclidean) classification can reduce accuracy below the better-modelled subset.

---

### F164 — Emergent: closed-form predictor recovers 58% of simulation accuracy

**Discovery (Part C):** Best closed-form (70.3%) ÷ simulation (96.9%) = **58%**. The other 42% requires the full mixed-feature distribution, not its first moment under any per-feature linear correction.

This is the quantitative statement of "the composition attractor is a property of the feature-extraction operator, not feature-space geometry." The geometry-based closed-form has a hard ceiling at ~70%; the operator-based simulation hits 97%. The gap is irreducible under per-feature linear models.

---

### F165 — Emergent: closed-form errors are oscillator-biased; "still broken" pairs require nonlinear separation

**Discovery (Part D):** The closed-form predictor mislabels 27 pairs as oscillator. 21 of these were also wrong under raw midpoint (nb47); 7 are *new* errors created by the linear correction; 14 are pairs the linear correction *fixed* (from raw wrong → closed-form right). The "still broken" 21 form a cohort of pairs that no per-feature linear model can classify correctly — they need joint information across features.

**Implication:** A useful closed-form composition predictor would need either (a) per-pair information (i.e., not per-feature), or (b) a multivariate correction that captures cross-feature interactions in the mixed distribution. The simplest version of (b) would be a single 6×6 linear map `actual_vec = M · midpt_vec + c` instead of 6 independent regressions — left as a follow-up.

---

### Findings
F158–F165 added. Total findings: **165**.

**Closing the composition arc:**
- nb46 found a 43% empirical attractor (declining_osc).
- nb47 found a 45.3% midpoint-prediction ceiling and a 77.5% Voronoi paradox.
- nb48 found that simulation hits 96.9% and the nonlinearity expels mixtures from the oscillator basin (84%).
- nb49 finds that closed-form (per-feature linear correction) maxes at **70.3%**. The geometry-only path has a hard ceiling.

**Conclusion:** there is no closed-form predictor of the composition table under per-feature linear correction. The composition attractor cannot be reduced to "centroid geometry plus per-feature scaling and shift" — the joint nonlinearity is essential. Thread 1 closes negatively but informatively: the simulation remains the right tool for predicting mixture outcomes.

**Open follow-up (deferred):** does a multivariate (cross-feature) linear correction `actual = M · midpt + c` close the gap, or is the gap genuinely nonlinear (i.e., even a 6×6 affine map fails)? If the latter, that strengthens the case that learned embeddings (Thread 2 next) which can capture cross-feature structure should outperform either closed-form approach.
